In [1]:
from Data_query.spark_config import *
from visualisation import *
import numpy as np
spark.catalog.setCurrentDatabase("solar_analytics")
warehouse_dir = spark.conf.get("spark.sql.warehouse.dir")
print(warehouse_dir)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/15 06:30:44 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).
25/09/15 06:30:48 WARN ConfigurationHelper: Option fs.s3a.connection.establish.timeout is too low (5,000 ms). Setting to 15,000 ms instead
Hive Session ID = 7e3f058e-e7c3-4053-b42a-d9311c4d3ad7


s3a://project-ciccada/spark-warehouse


In [ ]:
meta = spark.read.table("meta_single_inverters_wrong_capacity_up2_3c")\
    .filter(col("wrong_capacity") == False).filter(col("ac_capacity_kw").isNotNull())

ts = spark.read.table("ts")
ts = ts.filter("is_pv = True").filter(hour(col("t_stamp")).between(0, 6) | hour(col("t_stamp")).between(22, 23))\
    .select("circuit_id", "t_stamp", "voltage").withColumnRenamed("circuit_id", "circuit_id_ts")

ts = ts.join(meta.select("circuit_id", "site_id",  "state"), ts.circuit_id_ts == meta.circuit_id, "inner").drop("circuit_id_ts")
ts = ts.filter("voltage <= 265 and voltage >= 180")
ts = ts.groupBy("site_id", "state", "t_stamp").agg(avg("voltage").alias("voltage"))
ts = ts.groupBy(